# Video2X (upscale lên 4K) trên Colab (T4)

Repo: https://github.com/k4yt3x/video2x (bản 6.x, README nêu 6.4.0 là mới nhất lúc đọc).

## Điểm KHÔNG chắc chắn (nói thẳng)
- Video2X cần **Vulkan** (README: GPU phải hỗ trợ Vulkan). Ở Colab, driver NVIDIA thường **chỉ có CUDA**, **có thể thiếu Vulkan ICD** (`nvidia_icd.json`). Tôi **không thể xác minh** điều này từ máy Local, notebook sẽ **kiểm tra thật** ở ô 3.
- Docs Video2X nêu `vkEnumeratePhysicalDevices failed -3` thì thêm `--privileged` (chỉ áp dụng cho Docker, không dùng được trong Colab).
- Tên tệp AppImage trên trang Releases không đọc được tĩnh; notebook lấy danh sách asset qua GitHub API lúc chạy.
- Cú pháp CLI xác minh từ docs: `video2x -i in.mp4 -o out.mp4 -p realesrgan -s 4 --realesrgan-model realesr-animevideov3`; chọn GPU `-g`, liệt kê `--list-gpus`.

## Phương án dự phòng (tự động theo kết quả kiểm tra)
1. **Video2X AppImage** nếu có Vulkan.
2. **Real-ESRGAN bản PyTorch (CUDA, không cần Vulkan)** — chạy được trên T4. Chưa xác minh trọn vẹn trên Colab (`# TODO: kiểm tra`).
3. **FFmpeg lanczos** lên 3840x2160: luôn chạy được, nhưng không phải AI, chỉ phóng to.

> **Bước 0:** vào `Runtime > Change runtime type > T4 GPU` rồi bấm Save. Phiên Colab miễn phí có giới hạn thời gian; mọi kết quả đều được lưu ra Google Drive để không mất khi phiên bị ngắt.

Cấu trúc thư mục trên Drive (tự tạo ở ô mount):

```
MyDrive/content-factory/
  input/    <- đưa kịch bản .txt, video, audio lên đây
  output/   <- kết quả để tải về Local
  models/   <- trọng số / checkpoint
```

## 1. Cấu hình

In [ ]:
#@title Tham số
VIDEO_FILE = "video_thap.mp4"  #@param {type:"string"}
OUT_NAME   = "video_4k.mp4"    #@param {type:"string"}
PROCESSOR  = "realesrgan"      #@param ["realesrgan", "libplacebo"]
REALESRGAN_MODEL = "realesr-animevideov3"  #@param ["realesr-animevideov3", "realesrgan-plus"]
SCALE = 4                      #@param {type:"integer"}
TARGET_W, TARGET_H = 3840, 2160
FORCE_METHOD = "auto"          #@param ["auto", "video2x", "realesrgan_pytorch", "ffmpeg_lanczos"]

## 2. Mount Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/content-factory'
IN_DIR, OUT_DIR, MODEL_DIR = f'{BASE}/input', f'{BASE}/output', f'{BASE}/models'
for d in (IN_DIR, OUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)
print('Input :', IN_DIR)
print('Output:', OUT_DIR)
print('Models:', MODEL_DIR)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'KHONG THAY GPU. Hay chon Runtime > Change runtime type > T4 GPU.')

## 3. Kiểm tra Vulkan thật trên Colab

In [ ]:
import subprocess, glob, os
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return r

sh('apt-get -y -qq install libvulkan1 vulkan-tools mesa-vulkan-drivers ffmpeg')
icds = glob.glob('/usr/share/vulkan/icd.d/*.json') + glob.glob('/etc/vulkan/icd.d/*.json')
print('ICD tim thay:', icds)
has_nvidia_icd = any('nvidia' in p.lower() for p in icds)
r = sh('vulkaninfo --summary')
print(r.stdout[-2500:] or r.stderr[-1500:])
vk_gpu = ('NVIDIA' in r.stdout) and ('Tesla' in r.stdout or 'T4' in r.stdout or 'GPU' in r.stdout)
VULKAN_OK = r.returncode == 0 and 'NVIDIA' in r.stdout
print('\n>>> Vulkan tren GPU NVIDIA:', 'CO' if VULKAN_OK else 'KHONG (hoac chi co llvmpipe CPU)')
# TODO: kiem tra - thu tao ICD thu cong neu thieu: tao /etc/vulkan/icd.d/nvidia_icd.json tro toi libGLX_nvidia.so.0.
# Khong ep ghi o day vi chua xac minh tren Colab; neu can hay lam thu cong roi chay lai o nay.

## 4A. Phương án chính: Video2X AppImage (nếu có Vulkan)

In [ ]:
import requests, os, subprocess, shutil
USE_V2X = (FORCE_METHOD == 'video2x') or (FORCE_METHOD == 'auto' and VULKAN_OK)
V2X = '/content/Video2X.AppImage'
if not USE_V2X:
    print('Bo qua Video2X (khong co Vulkan NVIDIA hoac FORCE_METHOD khac).')
else:
    try:
        rel = requests.get('https://api.github.com/repos/k4yt3x/video2x/releases/latest', timeout=30).json()
        print('Ban phat hanh:', rel.get('tag_name'))
        assets = [a for a in rel['assets'] if a['name'].lower().endswith('.appimage')]
        print('AppImage:', [a['name'] for a in assets])
        a = assets[0]
        with requests.get(a['browser_download_url'], stream=True, timeout=600) as rr:
            rr.raise_for_status()
            with open(V2X, 'wb') as f:
                shutil.copyfileobj(rr.raw, f)
        os.chmod(V2X, 0o755)
        print(subprocess.run([V2X, '--appimage-extract-and-run', '--help'], capture_output=True, text=True).stdout[:1500])
        # TODO: kiem tra - '--appimage-extract-and-run' (co the khong can FUSE tren Colab)
    except Exception as e:
        print('LOI tai/chay AppImage:', e)
        USE_V2X = False

In [ ]:
import subprocess, os, shutil
RESULT = None
src = f'{IN_DIR}/{VIDEO_FILE}'
assert os.path.exists(src), f'Khong thay {src}. Hay dua video len Drive/input.'
shutil.copy2(src, '/content/in.mp4')
tmp_out = '/content/out_v2x.mp4'
if USE_V2X:
    # Cu phap xac minh tu docs: video2x -i -o -p realesrgan -s 4 --realesrgan-model <ten> ; libplacebo dung -w -h
    if PROCESSOR == 'realesrgan':
        cmd = [V2X, '--appimage-extract-and-run', '-i', '/content/in.mp4', '-o', tmp_out,
               '-p', 'realesrgan', '-s', str(SCALE), '--realesrgan-model', REALESRGAN_MODEL]
    else:
        cmd = [V2X, '--appimage-extract-and-run', '-i', '/content/in.mp4', '-o', tmp_out,
               '-w', str(TARGET_W), '-h', str(TARGET_H), '-p', 'libplacebo',
               '--libplacebo-shader', 'anime4k-v4-a+a']
    print(' '.join(cmd))
    r = subprocess.run(cmd, text=True, capture_output=True)
    print(r.stdout[-2000:])
    if r.returncode == 0 and os.path.exists(tmp_out):
        RESULT = tmp_out
    else:
        print('LOI Video2X:', r.stderr[-3000:])
        print('>> Chuyen sang phuong an du phong o muc 4B.')

## 4B. Dự phòng: Real-ESRGAN PyTorch (CUDA) rồi FFmpeg
Chạy khi Video2X không dùng được. Real-ESRGAN gốc: https://github.com/xinntao/Real-ESRGAN . Gói `basicsr` đôi khi lỗi với torchvision mới (**TODO: kiểm tra**, có thể phải vá `functional_tensor`).

In [ ]:
import subprocess, os, shutil
if RESULT is None and FORCE_METHOD in ('auto', 'realesrgan_pytorch'):
    def run(cmd, cwd=None):
        r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
        print('$', cmd); print(r.stdout[-1200:]);
        if r.returncode: print('LOI:', r.stderr[-1500:])
        return r.returncode == 0
    ok = run('git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN') if not os.path.isdir('/content/Real-ESRGAN') else True
    ok = ok and run('pip install -q basicsr facexlib gfpgan', '/content/Real-ESRGAN')
    ok = ok and run('pip install -q -r requirements.txt', '/content/Real-ESRGAN')
    ok = ok and run('python setup.py develop', '/content/Real-ESRGAN')
    # TODO: kiem tra - ten co va mo hinh cua inference_realesrgan_video.py (-n, -s, --suffix, --fp32)
    ok = ok and run(f'python inference_realesrgan_video.py -n RealESRGAN_x4plus -i /content/in.mp4 -o /content/re_out -s {SCALE}', '/content/Real-ESRGAN')
    if ok:
        import glob
        outs = glob.glob('/content/re_out/*.mp4')
        RESULT = outs[0] if outs else None
    if RESULT is None:
        print('Real-ESRGAN PyTorch khong chay duoc -> dung FFmpeg lanczos o muc 4C.')

## 4C. Cuối cùng: ép đúng 3840x2160 và lưu ra Drive/output
Nếu chưa có kết quả AI, bước này dùng FFmpeg lanczos (chỉ phóng to, không phải AI).

In [ ]:
import subprocess, os, shutil
src_for_final = RESULT or '/content/in.mp4'
if RESULT is None:
    print('CANH BAO: chua co ket qua AI, chi phong to bang lanczos.')
dst = f'{OUT_DIR}/{OUT_NAME}'
vf = (f'scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=decrease:flags=lanczos,'
      f'pad={TARGET_W}:{TARGET_H}:(ow-iw)/2:(oh-ih)/2')
r = subprocess.run(['ffmpeg', '-y', '-i', src_for_final, '-i', '/content/in.mp4', '-map', '0:v:0', '-map', '1:a:0?',
                    '-vf', vf, '-c:v', 'libx264', '-crf', '18', '-preset', 'medium',
                    '-pix_fmt', 'yuv420p', '-c:a', 'copy', dst], text=True, capture_output=True)
if r.returncode != 0:
    print('LOI FFmpeg:', r.stderr[-3000:])
else:
    print('KET QUA:', dst, f'({os.path.getsize(dst)/1e6:.1f} MB)')
    print(subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
                          'stream=width,height', '-of', 'csv=p=0', dst], text=True, capture_output=True).stdout)